# 02 - Pre-processamento de Dados

## Objetivo
Preparar os dados para modelagem com qualidade e reprodutibilidade.

Neste notebook vamos:
1. Carregar o dataset
2. Remover colunas nao uteis
3. Codificar a variavel alvo
4. Separar features e alvo
5. Fazer split treino/teste com estratificacao
6. Padronizar variaveis numericas para modelos sensiveis em relação a escala
7. Salvar os artefatos processados

In [1]:
import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 50)

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 1. Carga dos dados

In [2]:
dataset_path = kagglehub.dataset_download('uciml/breast-cancer-wisconsin-data')
csv_path = os.path.join(dataset_path, 'data.csv')

df = pd.read_csv(csv_path)

print(f'Arquivo carregado: {csv_path}')
print(f'Shape original: {df.shape}')
df.head()

Arquivo carregado: C:\Users\guirr\.cache\kagglehub\datasets\uciml\breast-cancer-wisconsin-data\versions\2\data.csv
Shape original: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## 2. Limpeza basica

Regras aplicadas:
- Remover `Unnamed: 32` (coluna vazia)
- Remover `id` (sem relevância nestes dados clínicos)

In [3]:
colunas_remover = ['Unnamed: 32', 'id']
df_limpo = df.drop(columns=colunas_remover, errors='ignore').copy()

print(f'Shape apos limpeza: {df_limpo.shape}')
print('Colunas removidas:', colunas_remover)
print('Colunas restantes:', df_limpo.shape[1])

Shape apos limpeza: (569, 31)
Colunas removidas: ['Unnamed: 32', 'id']
Colunas restantes: 31


## 3. Codificacao da variavel alvo

Mapeamento:
- `M` (maligno) -> 1
- `B` (benigno) -> 0

In [4]:
mapa_alvo = {'M': 1, 'B': 0}
df_limpo['diagnosis_encoded'] = df_limpo['diagnosis'].map(mapa_alvo)

if df_limpo['diagnosis_encoded'].isnull().any():
    raise ValueError('Existem valores inesperados na coluna diagnosis.')

print('Distribuicao da classe original:')
print(df_limpo['diagnosis'].value_counts())
print('\nDistribuicao da classe codificada:')
print(df_limpo['diagnosis_encoded'].value_counts())

Distribuicao da classe original:
diagnosis
B    357
M    212
Name: count, dtype: int64

Distribuicao da classe codificada:
diagnosis_encoded
0    357
1    212
Name: count, dtype: int64


## 4. Separacao de X e y

In [5]:
X = df_limpo.drop(columns=['diagnosis', 'diagnosis_encoded'])
y = df_limpo['diagnosis_encoded']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (569, 30)
y shape: (569,)


## 5. Split treino/teste (estratificado)

Usamos estratificacao para manter a mesma proporcao de classes em treino e teste.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} | y_test: {y_test.shape}')

print('\nProporcao da classe positiva (1=Maligno):')
print(f'Treino: {y_train.mean():.3f}')
print(f'Teste:  {y_test.mean():.3f}')

X_train: (455, 30) | X_test: (114, 30)
y_train: (455,) | y_test: (114,)

Proporcao da classe positiva (1=Maligno):
Treino: 0.374
Teste:  0.368


## 6. Padronizacao (StandardScaler)

Importante para algoritmos sensiveis a escala, como KNN e Regressao Logistica.
A Arvore de Decisao pode usar dados sem escala.

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print('Padronizacao concluida.')
print(f'X_train_scaled shape: {X_train_scaled_df.shape}')
print(f'X_test_scaled shape: {X_test_scaled_df.shape}')

Padronizacao concluida.
X_train_scaled shape: (455, 30)
X_test_scaled shape: (114, 30)


## 7. Salvando artefatos processados

Vamos salvar:
- **base limpa completa** — gerada na etapa 2 (limpeza) e etapa 3 (codificacao do alvo)
- **treino/teste sem escala** — gerados na etapa 5 (split treino/teste)
- **treino/teste com escala** — gerados na etapa 6 (padronizacao com StandardScaler)
- **y treino/teste** — gerados na etapa 5 (split treino/teste)


In [8]:
saida = '../data/processed'
os.makedirs(saida, exist_ok=True)

df_limpo.to_csv(f'{saida}/breast_cancer_limpo.csv', index=False)

X_train.to_csv(f'{saida}/X_train_raw.csv', index=False)
X_test.to_csv(f'{saida}/X_test_raw.csv', index=False)
X_train_scaled_df.to_csv(f'{saida}/X_train_scaled.csv', index=False)
X_test_scaled_df.to_csv(f'{saida}/X_test_scaled.csv', index=False)
y_train.to_csv(f'{saida}/y_train.csv', index=False)
y_test.to_csv(f'{saida}/y_test.csv', index=False)

print('Arquivos salvos com sucesso em data/processed:')
for nome in [
    'breast_cancer_limpo.csv',
    'X_train_raw.csv', 'X_test_raw.csv',
    'X_train_scaled.csv', 'X_test_scaled.csv',
    'y_train.csv', 'y_test.csv'
]:
    print('-', nome)

Arquivos salvos com sucesso em data/processed:
- breast_cancer_limpo.csv
- X_train_raw.csv
- X_test_raw.csv
- X_train_scaled.csv
- X_test_scaled.csv
- y_train.csv
- y_test.csv


## Conclusao

Pre-processamento finalizado.

Proximo notebook: `modelagem.ipynb`, onde vamos treinar e comparar os modelos abaixo:
- Regressao Logistica
- Arvore de Decisao
- KNN